#### This file contains the classes/functions used in the main file

In [1]:
import hxtorch.snn as hxsnn
import torch
import hxtorch
   # help(hxsnn.Synapse)
   # help(hxsnn.LIF) the notebook uses a older hxtorch version so the Github code cannot be 
    # copied 1:1
from hxtorch.snn.transforms.weight_transforms import linear_saturating
from functools import partial

In [2]:
%matplotlib inline
from _static.tutorial.hxsnn_intro_plots import plot_training, plot_compare_traces, plot_mock, plot_scaled_trace, plot_targets
from _static.common.helpers import setup_hardware_client, save_nightly_calibration
setup_hardware_client()

INFO  15:09:04,596  demo_helpers Connection to hxcube7fpga3chip61_1 established


In [3]:
import inspect
import hxtorch.snn as hxsnn
print(inspect.signature(hxsnn.LIF))
print("\n     ")
print(inspect.signature(hxsnn.Synapse))

(size: 'int', experiment: 'Experiment', leak: 'ModuleParameterType' = 80, reset: 'ModuleParameterType' = 80, threshold: 'ModuleParameterType' = 125, tau_mem: 'ModuleParameterType' = 1e-05, tau_syn: 'ModuleParameterType' = 1e-05, i_synin_gm: 'ModuleParameterType' = 500, membrane_capacitance: 'ModuleParameterType' = HXTransformedModelParameter(), refractory_time: 'ModuleParameterType' = 1e-06, synapse_dac_bias: 'ModuleParameterType' = 600, holdoff_time: 'ModuleParameterType' = 0.0, method: 'str' = 'superspike', alpha: 'float' = 50.0, execution_instance: 'Optional[ExecutionInstance]' = None, enable_spike_recording: 'bool' = True, enable_cadc_recording: 'bool' = True, enable_cadc_recording_placement_in_dram: 'bool' = False, enable_madc_recording: 'bool' = False, record_neuron_id: 'Optional[int]' = None, placement_constraint: 'Optional[List[halco.LogicalNeuronOnDLS]]' = None, trace_offset: 'Union[Dict[halco.AtomicNeuronOnDLS, float], torch.Tensor, float]' = 0.0, trace_scale: 'Union[Dict[hal

In [7]:
help(linear_saturating)

Help on function linear_saturating in module hxtorch.spiking.transforms.weight_transforms:

linear_saturating(weight: torch.Tensor, scale: float = 1.0, min_weight: float = -63.0, max_weight: float = 63.0) -> torch.Tensor
    Scale all weights according to:
    
        w <- clip(scale * w, min_weight, max_weight)
    
    TODO: Maybe make this function member of HXSynapse and allow different
          weight transformations by inheritance.
    
    :param weight: The weight tensor to be transformed.
    :param scale: A constant the weight tensor is scaled with.
    :param min_weight: The minimum value, smaller values are clipped to after
        scaling.
    :param max_weight: The maximum value, bigger values are clipped to after
        scaling.
    
    :returns: The transformed weight tensor.



In [9]:
# help(hxsnn.MixedHXModelParameter) not helpful

#### hxtorch moch=True, training and execution with a premade calibration file

In [4]:
tau_mem = 10e-5        # membrane time constant (help(hxsnn.LIF) default)
tau_syn = 10e-5        # synaptic time constant (default)
weight_scale = 1.0     # to be measured with the alignment, but later !
model_threshold, bss2_threshold = 0.3, 125   # roels threshold / notebook value
model_leak, bss2_leak = 0., 80
model_reset, bss2_reset = 0., 80

# for hardware the exact number is not so relevant as the alignment later reveals the
# ratio

exp = hxsnn.Experiment(dt=1e-6, mock=True)
# to avoid potentially time consuming just-in-time calibration
exp.default_execution_instance.load_calib("spiking_calix-native.pkl") 

def one_lif(size, exp):
    return hxsnn.LIF(size=size, experiment=exp, tau_mem=tau_mem, tau_syn=tau_syn,
                     leak=hxsnn.MixedHXModelParameter(model_leak, bss2_leak),
                     reset=hxsnn.MixedHXModelParameter(model_reset, bss2_reset),
                     threshold=hxsnn.MixedHXModelParameter(model_threshold, bss2_threshold))

def one_syn(n_in, n_out):
    return hxsnn.Synapse(in_features=n_in, out_features=n_out, experiment=exp,
                         transform=partial(linear_saturating, scale=weight_scale))

# input 128 -> 64 -> 64 -> 64 -> 20
syn1, lif1 = one_syn(128, 64), one_lif(64)
syn2, lif2 = one_syn(64, 64),  one_lif(64)
syn3, lif3 = one_syn(64, 64),  one_lif(64)
syn4, lif4 = one_syn(64, 20),  one_lif(20)


inputs = (torch.rand((100, 4, 128)) < 0.05).float() # asked claude for a random signal

# like deep learning
z1 = lif1(syn1(hxsnn.LIFObservables(spikes=inputs)))
z2 = lif2(syn2(z1))
z3 = lif3(syn3(z2))
z4 = lif4(syn4(z3))

hxsnn.run(exp, inputs.shape[0])

print(z4.spikes.shape)          # expected: (100, 4, 20) = (time, batch, output neurons)
print(z4.spikes.sum(dim=0))     # spike count per sample and output neuron

torch.Size([100, 4, 20])
tensor([[ 0., 21.,  0.,  0., 17., 22., 17., 17., 33.,  0.,  0.,  0.,  0., 10.,
         15.,  4.,  5.,  0., 14., 32.],
        [ 0., 21.,  4.,  0., 22., 27., 16.,  0., 32.,  0.,  5.,  0.,  0., 17.,
         24.,  0.,  0.,  0.,  0., 19.],
        [ 0.,  9.,  0.,  0.,  3.,  0., 30.,  5., 33.,  0.,  0.,  0.,  0.,  0.,
         28., 16.,  0.,  1., 18., 29.],
        [ 2., 25.,  0.,  0.,  0.,  0.,  5.,  0., 34.,  0.,  0.,  0.,  0., 22.,
          0.,  5.,  0.,  0.,  0., 31.]], grad_fn=<SumBackward1>)


- I tested the difference when not having the calibration file (mock=False)

In [ ]:
exp1 = hxsnn.Experiment(dt=1e-6, mock=False)

def one_lif(size):
    return hxsnn.LIF(size=size, experiment=exp1, tau_mem=tau_mem, tau_syn=tau_syn,
                     leak=hxsnn.MixedHXModelParameter(model_leak, bss2_leak),
                     reset=hxsnn.MixedHXModelParameter(model_reset, bss2_reset),
                     threshold=hxsnn.MixedHXModelParameter(model_threshold, bss2_threshold))

def one_syn(n_in, n_out):
    return hxsnn.Synapse(in_features=n_in, out_features=n_out, experiment=exp1,
                         transform=partial(linear_saturating, scale=weight_scale))

inputs = (torch.rand((100, 4, 128)) < 0.05).float()

# input 128 -> 64 -> 64 -> 64 -> 20
syn11, lif11 = one_syn(128, 64), one_lif(64)
syn21, lif21 = one_syn(64, 64),  one_lif(64)
syn31, lif31 = one_syn(64, 64),  one_lif(64)
syn41, lif41 = one_syn(64, 20),  one_lif(20)

# like deep learning
z11 = lif11(syn11(hxsnn.LIFObservables(spikes=inputs)))
z21 = lif21(syn21(z11))
z31 = lif31(syn31(z21))
z41 = lif41(syn41(z31))

hxsnn.run(exp1, inputs.shape[0])

print(z41.spikes.shape)          # expected: (100, 4, 20) = (time, batch, output neurons)
print(z41.spikes.sum(dim=0))     # spike count per sample and output neuron

WARN  15:32:39,209  calix.common.base.check_values Parameter tau_mem was chosen at [100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.
 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100. 100.

#### Todo
- check how calibration works: ask simon whether its necessary to do the calibration myself
- alignment of hardware values to software values (tutorial)
- partitioning the dataset